In [1]:
from langchain_groq import ChatGroq
from langchain.agents import AgentExecutor, create_tool_calling_agent
from langchain_core.tools import tool
from langchain.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.output_parsers import StrOutputParser
import base64
import cv2
import pytesseract
from langgraph.graph import StateGraph, START, END
from typing_extensions import TypedDict, Any
import numpy as np
import os

c:\Users\rohit\miniconda3\envs\Genai\lib\site-packages\torch\utils\_pytree.py:185: FutureWarning: optree is installed but the version is too old to support PyTorch Dynamo in C++ pytree. C++ pytree support is disabled. Please consider upgrading optree using `python3 -m pip install --upgrade 'optree>=0.13.0'`.
  warnings.warn(


In [10]:
def encode_image(image_path):
    with open(image_path, "rb") as image_file:
        return base64.b64encode(image_file.read()).decode('utf-8')

In [23]:
class Obstacle_handler(TypedDict):
    image_path:str 
    button_text_list: list
    to_be_matched: str
    index:int
    catalogue_image_paths: list


def save_grid_layout(crops_list: list, file_name: str):

    folder_name="Test_image_folder"


    row1 = cv2.hconcat([crops_list[0], crops_list[1]])
    row2 = cv2.hconcat([crops_list[2], crops_list[3]])
    row3 = cv2.hconcat([crops_list[4], crops_list[5]])
    row4 = cv2.hconcat([crops_list[6], crops_list[7]])
    
    final_grid = cv2.vconcat([row1, row2, row3, row4])
    
    cv2.imwrite(os.path.join(folder_name, file_name), final_grid)
def find_buttons_and_text(state: Obstacle_handler) -> Obstacle_handler:
    result = model_button.predict(state["image_path"])
    button_class_id = next((i for i, n in model_button.names.items() if "button" in n.lower()), None)

    img = cv2.imread(state["image_path"])
    texts = []       
    button_crops = [] 
    saved_catalogues = []
    
    target_size = (300, 300)
    grid_index = 0

    for box in result[0].boxes:
        if int(box.cls[0]) == button_class_id and float(box.conf[0]) > 0.37:
            x1, y1, x2, y2 = map(int, box.xyxy[0])
            
            crop = img[y1:y2, x1:x2]
            if crop.size > 0:
                resized_crop = cv2.resize(crop, target_size)
                button_crops.append(resized_crop)
            
            if len(button_crops) == 8:
                output_name = f"images_{grid_index}.jpg"
                save_grid_layout(button_crops, output_name)
                saved_catalogues.append(output_name)
                
                button_crops = []
                grid_index += 1

    if len(button_crops) > 0:
        blank_placeholder = np.zeros((target_size[1], target_size[0], 3), dtype=np.uint8)
        
        while len(button_crops) < 8:
            button_crops.append(blank_placeholder)
            
        output_name = f"images_{grid_index}.jpg"
        save_grid_layout(button_crops, output_name)
        saved_catalogues.append(output_name)

    state["button_text_list"] = texts
    state["catalogue_image_paths"] = saved_catalogues 
    return state




def find_relevant_text(state: Obstacle_handler)->Obstacle_handler:


    print ("===============Insidet find relevant text=================")
    prompt= ChatPromptTemplate.from_messages([("system", """
    You are a intelligent human vision model, who has to decide what image in the given catalogue of image, exactly matches with the given *button text* by the user.

    Note that, the *button text* has to exactly match with the image if exists in the catalogue.
    If present then, return the index number from start,

    For Example :

        -> If the user given *button text* is exactly matching with the image present at 0th index:
        Output: 0   
        -> If the user given *button text* is exactly matching with the image present at 1st index:
        Output: 1
        -> If the user given *button text* is exactly matching with the image present at 2nd index:
        Output: 2  
        -> If the user given *button text* is exactly matching with the image present at 3rd index:
        Output: 3  
        -> If the user given *button text* is exactly matching with the image present at 4th index:
        Output:4                                         
        -> If the user given *button text* is exactly matching with the image present at 5th index:
        Output: 5  
        -> If the user given *button text* is exactly matching with the image present at 6th index:
        Output: 6  
        -> If the user given *button text* is exactly matching with the image present at 7th index:
        Output: 7  
        -> If the user given *button text* does not exactly match with any image in the catalogue:
        Output: None  


    Note:

        - Do not explain.
        - Do not add any extra text.
        - Do not add punctuation.
        - Do not add quotes.
        - Do not add markdown.
        - Do not describe your reasoning.                                                                            
    """),
    ("human", [{
        "type":"text",
        "text": "Here is the *button text* from user: {button_text}"
    },
    {"type":"image", "image_url": {
        "url":"data:image/jpeg;base64,{base64_image}"}
    }])])

    chain=prompt | vlm_model | StrOutputParser()

    for _, dirs, files in os.walk("Test_image_folder"):

        
        for f in files:
            base64_image=encode_image(os.path.join("Test_image_folder", f))


            print ("Inside the lookup loop looking at: ", os.path.join("Test_image_folder", f))
            vlm_button_chain_output=chain.invoke({
                "button_text":state["to_be_matched"],
                "base64_image":base64_image
            })

            print ("Output_returned: ", vlm_button_chain_output)

            if "none" not in str(vlm_button_chain_output).lower():
                state["index"]=int(str(vlm_button_chain_output).strip())
            else:
                state["index"]=None
    
    
    return state




graph=StateGraph(Obstacle_handler)

graph.add_node("find_buttons_and_text", find_buttons_and_text)
graph.set_entry_point("find_buttons_and_text")
graph.add_node("find_relevant_text", find_relevant_text)

graph.add_edge("find_buttons_and_text", "find_relevant_text")
graph.add_edge("find_relevant_text", END)

app=graph.compile()

In [24]:
@tool
def is_obstacle_present(screen_summary: str) -> str:
    """
    LOW PRIORITY FALLBACK TOOL.

    Use ONLY when an obstacle exists and it is NOT:
    - Login popup
    - Signup popup
    - Authentication wall

    If login-related UI is present,
    use login_page instead.

    """

    image_path="image11.png"
    base64_image = encode_image(image_path)

    prompt = ChatPromptTemplate.from_messages([
        (
            "system",
            """
You are an expert UI obstacle detector.

Your task is to identify the SINGLE button that should be clicked to remove
an obstacle that is blocking the job application workflow.

An obstacle is:
- Cookie banner
- Newsletter modal
- Advertisement popup
- Chat widget covering content
- Consent dialog
- Any overlay preventing interaction with the main page

Assume an obstacle exists because this tool is only called after another
agent has determined that a blocker is present.

Instructions:
1. Identify the blocker.
2. Identify the best button for dismissing it.
3. Return ONLY the button text.

Preferred buttons:
Close
x
Dismiss
Not now
No thanks
Skip
Continue without signing in
Accept
Accept All
Reject 
Reject All

Output Rules:
- Return exactly one button label which is present in the screenshot.
- No explanations.
- No JSON.
- No markdown.
- No prefixes.
- No suffixes.
- Preserve the button text exactly as shown.
"""
        ),
        (
            "human",
            [
                {"type": "text", "text": "{input}"},
                {
                    "type": "image_url",
                    "image_url": {
                        "url": f"data:image/jpeg;base64,{base64_image}"
                    }
                }
            ]
        )
    ])

    chain = prompt | vlm_model | StrOutputParser()

    output=chain.invoke({
        "input": "Find the dismiss button for the blocking element."
    })
    graph_output=app.invoke({
        "image_path":image_path,
        "to_be_matched":"x"
    })

    #index=graph_output["index"]

    


    
    return output
                

    
@tool
def login_page(screen_summary:str)->None:
    """
    HIGH PRIORITY TOOL.

    Use this tool whenever the screen contains:
    - Sign in
    - Log in
    - Continue with Google
    - Continue with Email
    - Create Account
    - Sign Up

    If a login or signup dialog is present,
    ALWAYS use this tool instead of is_obstacle_present.

    """
    print ("LOGIN OBSTACLE")
    




@tool
def no_obstacle(screen_summary: str) -> str:
    """
    Use this tool ONLY when the job application workflow is accessible
    and no blocking UI element is present.

    Call this tool when:
    - An Apply button is visible.
    - A job description is visible.
    - An application form is visible.
    - Resume upload fields are accessible.
    - Submit/Next buttons are accessible.
    - The page can be interacted with normally.

    DO NOT call this tool when:
    - A popup covers the page.
    - A login wall blocks access.
    - A consent banner prevents interaction.
    - A modal or overlay must first be dismissed.

    Returns:
        Confirmation that the application flow can continue.
    """
    return "NO_OBSTACLE_PRESENT"

In [25]:
tools=[is_obstacle_present, no_obstacle, login_page]


base64_image=encode_image("image11.png")
prompt=ChatPromptTemplate.from_messages([(
    "system", "You are an expert job seeking human, you have to apply to the job, you will be provided with the screenshot to find in the fields."
),
("human", [
    {"type":"text", "text":"Here is the job page screenshot"},
    {"type":"image_url", "image_url":{
        "url":"data:image/jpeg;base64,{base64_image}"
    }}
]), MessagesPlaceholder("agent_scratchpad")])
agent=create_tool_calling_agent(llm=vlm_model, tools=tools, prompt=prompt)


agent_exe=AgentExecutor(agent=agent, tools=tools, verbose=True, max_iterations=5)

In [26]:
result=agent_exe.invoke({
    "base64_image":base64_image
})



> Entering new AgentExecutor chain...

Invoking: `is_obstacle_present` with `{'screen_summary': 'The screenshot shows a dark screen with a possible cookie consent banner at the bottom, which may be blocking access to the job application.'}`



image 1/1 c:\Users\rohit\Desktop\All_Language_programs\Linkedin_Automation_Proj\image11.png: 448x640 9 AXButtons, 1 AXImage, 1 AXLink, 126.2ms
Speed: 11.9ms preprocess, 126.2ms inference, 20.2ms postprocess per image at shape (1, 3, 448, 640)
===============Insidet find relevant text=================
Inside the lookup loop looking at:  Test_image_folder\images_0.jpg
Output_returned:  None
No thanks
Invoking: `no_obstacle` with `{'screen_summary': 'The job application workflow is accessible and no blocking UI element is present.'}`


NO_OBSTACLE_PRESENT
Invoking: `is_obstacle_present` with `{'screen_summary': 'The screenshot shows a dark screen with a possible cookie consent banner at the bottom.'}`



image 1/1 c:\Users\rohit\Desktop\All_Langua

In [23]:
result

{'base64_image': '...',
 'output': '{\n  "response": "NO_OBSTACLE_PRESENT"\n}'}

In [9]:
class test_state(TypedDict):
    path:str
    submit:str
    driver:Any
    move:str
    last_page:str
    handle:str
    form_done:str

In [10]:
def take_ss(state:test_state)->test_state:
    state["driver"].save_screenshot("ss.png")
    return state

In [ ]:
def move_down (state:test_state)->test_state:

    state["driver"].execute_script(
        "window.scrollBy(0, window.innerHeight * 0.9);"
    )
    if driver.execute_script(
        "return window.innerHeight + window.pageYOffset >= document.body.scrollHeight - 10"
    ):
       state["last_page"]="yes"

    os.remove(state["path"])
    state["driver"].save_screenshot("visible.png")

    return state

Duplicate

In [19]:
ans=model_button.predict("image11.png")


image 1/1 c:\Users\rohit\Desktop\All_Language_programs\Linkedin_Automation_Proj\image11.png: 448x640 9 AXButtons, 1 AXImage, 1 AXLink, 172.3ms
Speed: 104.1ms preprocess, 172.3ms inference, 10.1ms postprocess per image at shape (1, 3, 448, 640)


In [20]:
ans[0].show()

In [ ]:
class Obstacle_handler(TypedDict):
    image_path:str 
    button_text_list: list
    to_be_matched: str
    index:int

def find_buttons_and_text(state:Obstacle_handler)->Obstacle_handler:
    result = model_button.predict(state["image_path"])
    button_class_id = next((i for i, n in model_button.names.items() if "button" in n.lower()), None)

    img = cv2.imread(state["image_path"])
    texts = []

    for box in result[0].boxes:

        if int(box.cls[0]) == button_class_id and float(box.conf[0]) > 0.37: #<---- Alter with the confidence score threshold to change the values.

            x1, y1, x2, y2 = map(int, box.xyxy[0])

            crop = img[y1:y2, x1:x2]

            gray = cv2.cvtColor(crop, cv2.COLOR_BGR2GRAY)
            cv2.imwrite(f"Test_folder/a_{y1+y2}.png", gray)
            text = pytesseract.image_to_string(
                gray,
                config="--oem 3 --psm 7"
            ).strip()   

            if text:
                texts.append(text)

    state["button_text_list"]=texts
    return state


def find_relevant_text(state: Obstacle_handler)->Obstacle_handler:

    # text=state["to_be_matched"]
    # matches=difflib.get_close_matches(text, state["button_text_list"], n=1)

    # print (state["to_be_matched"], matches, state["button_text_list"])
    # match_indices = [state["button_text_list"].index(match) for match in matches]
    # print (match_indices)
    # state["index"]=[0]


    prompt= ChatPromptTemplate.from_messages([("system", """
    You are a intelligent human vision model, who has to decide what image in the given catalogue of image, exactly matches with the given *button text* by the user.

    Note that, the *button text* has to exactly match with the image if exists in the catalogue.
    If present then, return the index number from start,

    For Example :

        -> If the user given *button text* is exactly matching with the image present at 0th index:
        Output: 0   
        -> If the user given *button text* is exactly matching with the image present at 1st index:
        Output: 1
        -> If the user given *button text* is exactly matching with the image present at 2nd index:
        Output: 2  
        -> If the user given *button text* is exactly matching with the image present at 3rd index:
        Output: 3  
        -> If the user given *button text* is exactly matching with the image present at 4th index:
        Output:4                                         
        -> If the user given *button text* is exactly matching with the image present at 5th index:
        Output: 5  
        -> If the user given *button text* is exactly matching with the image present at 6th index:
        Output: 6  
        -> If the user given *button text* is exactly matching with the image present at 7th index:
        Output: 7  
        -> If the user given *button text* does not exactly match with any image in the catalogue:
        Output: None                                                                              
    """),
    ("human", [{
        "type":"text",
        "text": "Here is the *button text* from user: {button_text}"
    },
    {"type":"image", "image_url": {
        "url":f"data:image/jpeg;base64,{base64_image}"}
    }])])

    base64_image=encode_image()
    return state




graph=StateGraph(Obstacle_handler)

graph.add_node("find_buttons_and_text", find_buttons_and_text)
graph.set_entry_point("find_buttons_and_text")
graph.add_node("find_relevant_text", find_relevant_text)

graph.add_edge("find_buttons_and_text", "find_relevant_text")
graph.add_edge("find_relevant_text", END)

app=graph.compile()

In [ ]:
@tool
def is_obstacle_present(screen_summary: str) -> str:
    """
    LOW PRIORITY FALLBACK TOOL.

    Use ONLY when an obstacle exists and it is NOT:
    - Login popup
    - Signup popup
    - Authentication wall

    If login-related UI is present,
    use login_page instead.

    """

    image_path="image11.png"
    base64_image = encode_image(image_path)

    prompt = ChatPromptTemplate.from_messages([
        (
            "system",
            """
You are an expert UI obstacle detector.

Your task is to identify the SINGLE button that should be clicked to remove
an obstacle that is blocking the job application workflow.

An obstacle is:
- Login popup
- Sign-up popup
- Cookie banner
- Newsletter modal
- Advertisement popup
- Chat widget covering content
- Consent dialog
- Any overlay preventing interaction with the main page

Assume an obstacle exists because this tool is only called after another
agent has determined that a blocker is present.

Instructions:
1. Identify the blocker.
2. Identify the best button for dismissing it.
3. Return ONLY the button text.

Preferred buttons:
Close
x
Dismiss
Not now
No thanks
Skip
Continue without signing in
Accept
Accept All

Output Rules:
- Return exactly one button label.
- No explanations.
- No JSON.
- No markdown.
- No prefixes.
- No suffixes.
- Preserve the button text exactly as shown.
"""
        ),
        (
            "human",
            [
                {"type": "text", "text": "{input}"},
                {
                    "type": "image_url",
                    "image_url": {
                        "url": f"data:image/jpeg;base64,{base64_image}"
                    }
                }
            ]
        )
    ])

    chain = prompt | vlm_model | StrOutputParser()

    output=chain.invoke({
        "input": "Find the dismiss button for the blocking element."
    })
    graph_output=app.invoke({
        "image_path":image_path,
        "to_be_matched":"x"
    })

    #index=graph_output["index"]

    


    
    return output
                

    
@tool
def login_page(screen_summary:str)->None:
    """
    HIGH PRIORITY TOOL.

    Use this tool whenever the screen contains:
    - Sign in
    - Log in
    - Continue with Google
    - Continue with Email
    - Create Account
    - Sign Up

    If a login or signup dialog is present,
    ALWAYS use this tool instead of is_obstacle_present.

    """
    print ("LOGIN OBSTACLE")
    




@tool
def no_obstacle(screen_summary: str) -> str:
    """
    Use this tool ONLY when the job application workflow is accessible
    and no blocking UI element is present.

    Call this tool when:
    - An Apply button is visible.
    - A job description is visible.
    - An application form is visible.
    - Resume upload fields are accessible.
    - Submit/Next buttons are accessible.
    - The page can be interacted with normally.

    DO NOT call this tool when:
    - A popup covers the page.
    - A login wall blocks access.
    - A consent banner prevents interaction.
    - A modal or overlay must first be dismissed.

    Returns:
        Confirmation that the application flow can continue.
    """
    return "NO_OBSTACLE_PRESENT"